# Семинар 5. Влияние AI-подсказок на разметку

Проект: определить интернет-цену товара по странице магазина.

## Цели

- собрать путь от URL до подсказки в интерфейсе;
- сравнить скорость и качество разметки без подсказки и с ней;
- проанализировать пользу и вред подсказок;

## Обозначения

- **Baseline** — люди не видят подсказку;
- **AI hint** — люди видят подсказку, но вводят ответ сами;
- **Gold** — проверенная цена по инструкции.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import sys, json, uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image as PILImage
from IPython.display import display, HTML, Markdown, Image

candidates = [folder / name for folder in [Path.cwd(), *Path.cwd().parents]
              for name in [".", "shad_seminar", "ysda-2026-seminar5-ai-hints"]]
ROOT = next((p.resolve() for p in candidates if (p / "src").is_dir() and (p / "data/seminar5_challenge/cards.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте папку семинара целиком: рядом с ноутбуком нужны src/ и data/.")
print("Python ядра:", sys.executable)
sys.path.insert(0, str(ROOT / "src"))
BUNDLE = ROOT / "data/seminar5_challenge"
from prompt_study import TASK_RULE
from seminar5_research import load_experiment, cluster_interval
ARMS = ["baseline", "assisted"]
LABELS = ["Baseline", "AI hint"]
COLORS = ["#6479AF", "#D98559"]
plt.rcParams.update({"figure.dpi":120, "font.size":11, "axes.spines.top":False, "axes.spines.right":False})
cards = pd.read_csv(BUNDLE / "cards.csv", dtype={"sample_id":str})
display(Markdown(f"В выборке **{len(cards)} карточек**. Gold хранится в `reference_price`, цена подсказки — в `hint_price`."))


# Часть I. Как устроен пайплайн

## Целевая цена

Нужна интернет-цена основного выбранного товара за одну продаваемую единицу для обычного
покупателя. Не учитываем карту, регистрацию, подписку, промокод, опт, рассрочку,
специальный способ оплаты и другую комплектацию.


In [ ]:
worker_instruction = (BUNDLE / "instruction.html").read_text()
display(HTML('<details><summary>Полная инструкция, которую видел исполнитель</summary>' + worker_instruction + '</details>'))
example = cards.set_index("sample_id").loc["7254e1e3decff0dd"]
display(Image(filename=str(ROOT / example.image_path), width=800))
display(Markdown("**Пример:** интернет-цена 47 990 ₽, цена в розничном магазине 54 300 ₽. Нужна интернет-цена."))


## Дизайн сравнения

Карточки, инструкция, оплата и перекрытие одинаковы. Группы исполнителей не пересекаются.
Единственное различие — наличие AI-подсказки.


In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("seminar5_%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
pool_settings = {
    "private_name": RUN_ID,
    "may_contain_adult_content": False,
    "will_expire": (datetime.now(timezone.utc) + timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%S"),
    "reward_per_assignment": 1.0,
    "assignment_max_duration_seconds": 600,
    "auto_accept_solutions": True,
    "mixer_config": {"real_tasks_count":1,"golden_tasks_count":0,"training_tasks_count":0},
    "defaults": {"default_overlap_for_new_task_suites":3},
}
baseline_project = json.loads((BUNDLE / "baseline_project.json").read_text())
display(pd.Series({"Перекрытие":3,"Оплата за ответ, ₽":1}).to_frame("Настройка"))

## От URL к изображению

Что отдаем модели?


In [ ]:
from playwright.async_api import async_playwright

async def screenshot_site(url, output_dir):
    """Открывает страницу и сохраняет видимую область в PNG."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    image_path = output_dir / f"page_{uuid.uuid4().hex[:8]}.png"

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch()
        try:
            page = await browser.new_page(viewport={"width": 1440, "height": 1400})
            response = await page.goto(url, wait_until="domcontentloaded", timeout=45000)
            if response is None or response.status >= 400:
                status = response.status if response else "нет ответа"
                raise RuntimeError(f"Страница недоступна: HTTP {status}")
            await page.wait_for_timeout(2000)
            await page.screenshot(path=str(image_path), full_page=False)
        finally:
            await browser.close()

    return image_path

In [ ]:
def prepare_vlm_input(image_path, prompt):
    """RGB-изображение и сообщение для локальной VLM; URL ей уже не нужен."""
    with PILImage.open(image_path) as source:
        picture = source.convert("RGB")
    messages = [{"role":"user", "content":[
        {"type":"image"},
        {"type":"text", "text":prompt},
    ]}]
    return picture, messages

### **URL страницы** → **Скриншот** → **VLM** → **Подсказка**


### Практика №1. Минимальный промпт VLM

Сформулируйте инструкцию для изображения. Выход — JSON с полями `price` и `status`.


In [ ]:
PROMPT_V1 = """
# TODO: сформулируйте цель, ограничения и формат JSON.
"""
print(PROMPT_V1)


In [ ]:
def render_hint(parsed):
    if parsed["status"] == "ok":
        amount = f'{parsed["price"]:,.2f}'.replace(",", " ").replace(".", ",")
        return f"Цена товара — {amount} ₽."
    if parsed["status"] == "no_price":
        return "На странице цена не указана."
    return "Не удалось определить цену."

## Подсказка в интерфейсе

В AI hint к `sample_id` и `source_url` добавляется поле `hint`. Человек вводит ответ сам.


### Как текст попадает в `input_values`

Payload одной карточки


In [ ]:
def make_assisted_tasks(rows, pool_id, overlap=3):
    """Добавляет подсказку в input_values каждого задания."""
    required = ["sample_id", "source_url", "hint"]

    tasks = []
    for row in rows[required].to_dict("records"):
        tasks.append({
            "pool_id": pool_id,
            "overlap": overlap,
            "input_values": {
                "sample_id": row["sample_id"],
                "source_url": row["source_url"],
                "hint": row["hint"],
            },
        })
    return tasks


example_tasks = cards.head(1)[["sample_id", "source_url"]].copy()
example_tasks["hint"] = render_hint({
    "status": "ok",
    "price": cards.iloc[0]["hint_price"],
})

payload = make_assisted_tasks(example_tasks, pool_id="demo_pool")
display(payload[0]["input_values"])

За внешний вид отвечает `view.alert`: тема `info` задает синее оформление, `label` — подпись, а `path` указывает на поле `hint`.


In [ ]:
hint_view = {
    "type": "view.alert",
    "theme": "info",
    "label": "AI-подсказка",
    "content": {
        "type": "view.text",
        "content": {"type": "data.input", "path": "hint"},
    },
}

In [ ]:
display(Image(filename=str(ROOT / "assets/ai_hint_ui_card.jpg"), width=450))

# Часть II. Анализ человеческой разметки

В каждой группе 100 одинаковых карточек и по три ответа на карточку.


In [ ]:
data, cards, metadata = load_experiment(BUNDLE)
counts = data.groupby(["sample_id","condition"]).size()
assert len(counts) == 2 * len(cards) and counts.eq(3).all()
assert data.groupby("worker").condition.nunique().eq(1).all()
assert not data.duplicated(["worker","sample_id"]).any()
design = data.groupby("condition").agg(
    Карточек=("sample_id","nunique"),
    Исполнителей=("worker","nunique"),
)
display(design.reindex(ARMS).rename(index=dict(zip(ARMS,LABELS))))


In [ ]:
display(Markdown("### Датасет ответов"))
column_guide = pd.DataFrame([
    ("response_id", "ID отдельного ответа"),
    ("sample_id", "ID карточки товара"),
    ("condition", "baseline или assisted"),
    ("worker", "ID исполнителя"),
    ("status", "статус ответа"),
    ("price", "цена, которую ввёл исполнитель"),
    ("duration_sec", "время выполнения в секундах"),
    ("source_clicked", "записанный клик по источнику"),
    ("reference_price", "gold-цена"),
    ("hint_price", "цена из AI-подсказки"),
    ("trap_type", "тип ловушки на странице"),
    ("numeric", "дан ли числовой ответ"),
    ("correct", "совпал ли ответ с gold"),
    ("hint_wrong", "была ли подсказка неверной"),
    ("matches_hint", "совпал ли ответ с ценой подсказки"),
], columns=["Столбец", "Что означает"])
display(column_guide)

## Время выполнения

$$\text{изменение времени}=\frac{T_{\mathrm{AI}}-T_{\mathrm{Baseline}}}{T_{\mathrm{Baseline}}}.$$

Плюс означает дольше, минус — быстрее. Bootstrap пересобирает исполнителей целиком.


In [ ]:
def plot_time_comparison(rows, suffix=""):
    median = rows.groupby("condition")["duration_sec"].median().reindex(ARMS)
    interval = cluster_interval(rows, "duration_sec")
    title_suffix = f" — {suffix}" if suffix else ""
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), layout="constrained")
    bars = axes[0].bar(LABELS, median, color=COLORS)
    axes[0].bar_label(bars, fmt="%.2f с", padding=4)
    axes[0].set(ylabel="Секунды", title="Медиана времени" + title_suffix, ylim=(0, median.max() * 1.25))
    axes[0].grid(axis="y", alpha=.25)
    axes[1].hlines(0, 100 * interval["low"], 100 * interval["high"], color=COLORS[0], lw=5)
    axes[1].scatter([100 * interval["effect"]], [0], s=90, color=COLORS[1], zorder=3)
    axes[1].axvline(0, color="black", ls="--", label="Нет изменения")
    axes[1].set(yticks=[], xlabel="Изменение медианы, %", title="95% кластерный bootstrap-интервал" + title_suffix)
    axes[1].grid(axis="x", alpha=.25); axes[1].legend()
    plt.show()
    return median, interval

median_time, time_result = plot_time_comparison(data)
time_change, time_low, time_high = time_result["effect"], time_result["low"], time_result["high"]
includes_zero = time_low <= 0 <= time_high
conclusion = ("Интервал включает 0: убедительного ускорения или замедления не установлено."
              if includes_zero else "Интервал целиком выше 0: есть свидетельство замедления."
              if time_low > 0 else "Интервал целиком ниже 0: есть свидетельство ускорения.")
display(Markdown(f"**Ответ:** {conclusion} Оценка {time_change:+.1%}, интервал [{time_low:+.1%}; {time_high:+.1%}]."))

In [ ]:
# Для каждого исполнителя считаем среднее время и долю числовых ответов.
worker_rows = data.dropna(subset=["duration_sec"]).copy()
worker_rows["gave_number"] = worker_rows["numeric"]
worker_summary = (worker_rows.groupby(["condition", "worker"])
                  .agg(answers=("response_id", "size"), mean_duration=("duration_sec", "mean"),
                       numeric_share=("gave_number", "mean"))
                  .reset_index())

# Кандидат на ручную проверку: среди самых быстрых 10% и без числовых ответов.
fast_cutoff = worker_summary["mean_duration"].quantile(.10)
suspicious_workers = worker_summary.loc[
    worker_summary["mean_duration"].le(fast_cutoff) & worker_summary["numeric_share"].eq(0)
].copy()

fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
for arm, label, color in zip(ARMS, LABELS, COLORS):
    arm_workers = worker_summary.loc[worker_summary["condition"].eq(arm)]
    ax.scatter(arm_workers["mean_duration"], 100 * arm_workers["numeric_share"],
               color=color, label=label, alpha=.7)
ax.scatter(suspicious_workers["mean_duration"], 100 * suspicious_workers["numeric_share"],
           facecolor="none", edgecolor="#C83E4D", s=140, linewidth=2.5)
ax.set(xlabel="Среднее время исполнителя, секунды", ylabel="Доля числовых ответов",
       title="Скорость и тип результата", ylim=(-5, 105))
ax.yaxis.set_major_formatter(PercentFormatter(100)); ax.grid(alpha=.25)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
plt.show()


# Все следующие анализы используют данные без ответов этого исполнителя.
suspicious_worker_ids = suspicious_workers["worker"]
data = data.loc[~data["worker"].isin(suspicious_worker_ids)].copy()
median_time_filtered = data.groupby("condition")["duration_sec"].median().reindex(ARMS)
time_after_filter = cluster_interval(data, "duration_sec")
fig, ax = plt.subplots(figsize=(9, 2.2), layout="constrained")
ax.hlines(0, 100 * time_after_filter["low"], 100 * time_after_filter["high"],
          color="#6479AF", linewidth=6)
ax.scatter(100 * time_after_filter["effect"], 0, color="#D98559", s=90, zorder=2)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set(yticks=[], xlabel="Изменение медианы времени, %",
       title="Время после фильтра: 95% кластерный bootstrap-интервал")
ax.grid(axis="x", alpha=.25); plt.show()
filtered_includes_zero = time_after_filter["low"] <= 0 <= time_after_filter["high"]
filtered_conclusion = ("Интервал включает 0: изменение времени по-прежнему неубедительно."
                       if filtered_includes_zero else "Интервал не включает 0.")
display(Markdown(
    f"**После фильтра осталось {len(data)} ответов.** Медиана времени: "
    f"{median_time_filtered['baseline']:.1f} с → {median_time_filtered['assisted']:.1f} с, "
    f"то есть время изменилось на {time_after_filter['effect']:+.1%}. {filtered_conclusion} "
    f"95% интервал "
    f"[{time_after_filter['low']:+.1%}; {time_after_filter['high']:+.1%}]."
))

## Качество показанных подсказок

Подсказки сравниваются с gold по карточкам.

In [ ]:
hint_is_correct = (cards["hint_price"] - cards["reference_price"]).abs() < .005
hint_counts = pd.Series({
    "Совпала с gold":int(hint_is_correct.sum()),
    "Не совпала с gold":int((~hint_is_correct).sum()),
})
hint_overview = hint_counts.to_frame("Карточек")
display(hint_overview.style.format({"Доля выборки":"{:.0%}"}))


## Совпадение с неверной подсказкой

Про типы ошибок:

| Ответ человека | Ответ верный? | Совпал с неверной подсказкой? | Что произошло |
|---:|:---:|:---:|---|
| **800 ₽** | **Нет** | **Да** | **Человек выбрал ошибку модели — этот случай нас интересует** |
| 700 ₽ | Нет | Нет | Человек ошибся иначе |
| Нет числового ответа | Нет | Нет | Человек не повторил конкретную цену модели |

В числитель попадает только выделенный случай: ответ человека равен конкретной неверной цене из подсказки.

### Что считаем

**Неверная подсказка → совпал ли ответ с ее ценой → доля совпадений по группам → (AI hint − Baseline)**

Сначала оставляем только карточки с `hint_wrong == True`. Для каждого ответа уже рассчитан признак `matches_hint`: он равен `True`, если цена человека совпала с ценой подсказки.

$$U_g=\frac{\text{ответы, равные неверной подсказке}}{\text{все ответы на карточках с неверной подсказкой}}.$$

Сначала отдельно получаем два значения: $U_{\mathrm{Baseline}}$ и $U_{\mathrm{AI\ hint}}$.

Знаменатель — **все ответы на этих карточках**.

В Baseline подсказку не показывали, но человек мог самостоятельно выбрать ту же заметную цену со страницы. Поэтому сравниваем группы:

$$\Delta U=100\,(U_{\mathrm{AI\ hint}}-U_{\mathrm{Baseline}})\quad\text{п.п.}$$

Положительное $\Delta U$ означает, что после показа подсказки люди чаще выбирали конкретную неверную цену модели.

### Практика №2

1. Оставьте ответы на карточках с неверной подсказкой.
2. Посчитайте среднее `matches_hint` отдельно для каждой группы.
3. ...

In [ ]:
# 1. Берём только карточки, на которых подсказка была неверной.
wrong = data.loc[
    ...
].copy()

# 2. Считаем долю ответов, равных конкретной неверной цене модели.
wrong_hint_match_rate = (
    wrong.groupby("condition")[...]
    .mean()
    .reindex(ARMS)
)

# 3. На сколько процентных пунктов эта доля выше после показа подсказки?
excess_wrong_hint_agreement_pp = 100 * (
    ...
)

print(
    "Совпадение с ошибочной подсказкой:",
    wrong_hint_match_rate,
    sep="\n",
)

print(
    "Дополнительное совпадение:",
    f"{excess_wrong_hint_agreement_pp:+.1f} п.п.",
)


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
rates_percent = 100 * wrong_hint_match_rate.reindex(ARMS)
labels = ["Подсказку\nне видели", "Подсказку\nвидели"]
bars = ax.bar(labels, rates_percent, color=COLORS)
ax.bar_label(bars, labels=[f"{value:.1f}%" for value in rates_percent], padding=5)
ax.set(ylabel="Ответ совпал с ошибкой модели", ylim=(0,max(35,rates_percent.max()*1.3)),
       title="Как часто человек выбирает конкретную неверную цену модели?")
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.grid(axis="y",alpha=.25)
plt.show()
display(Markdown(f"**Наблюдаемый ответ:** после показа ошибки совпадение выросло на **{excess_wrong_hint_agreement_pp:+.1f} п.п.**"))


In [ ]:
wrong_match_interval = cluster_interval(data,"matches_hint",subset="wrong")
np.testing.assert_allclose(
    100 * wrong_match_interval["effect"],
    excess_wrong_hint_agreement_pp,
)
effect_pp = excess_wrong_hint_agreement_pp
low_pp = 100 * wrong_match_interval["low"]
high_pp = 100 * wrong_match_interval["high"]

fig, ax = plt.subplots(figsize=(9,2.6))
ax.hlines(0,low_pp,high_pp,color=COLORS[0],lw=5)
ax.scatter(effect_pp,0,color=COLORS[1],s=90,zorder=3)
ax.axvline(0,color="black",ls="--",label="Нет изменения")
ax.set(yticks=[],xlabel="Изменение частоты выбора ошибки модели, п.п.",
       title=f"Оценка {effect_pp:+.1f}; 95% интервал [{low_pp:+.1f}; {high_pp:+.1f}]")
ax.grid(axis="x",alpha=.25); ax.legend(); plt.show()
inference = ("весь интервал выше нуля — показ ошибки убедительно увеличил частоту ее выбора"
             if low_pp > 0 else "интервал включает ноль — убедительное увеличение не установлено")
display(Markdown(f"**Ответ:** {inference}."))


## Польза верных и вред неверных подсказок

В предыдущем блоке мы смотрели только на неверные подсказки и проверяли, повторяет ли человек конкретную ошибку модели. Теперь нужен полный ответ: **что происходит с правильностью ответа человека, когда подсказка верна, и что происходит, когда она неверна?**

Важно не перепутать два признака:

| Признак | Что описывает | Значение `True` |
|---|---|---|
| `hint_wrong` | Качество подсказки модели | Подсказка не совпала с gold |
| `correct` | Качество ответа человека | Ответ человека совпал с gold |

Сначала делим ответы **по качеству подсказки**, а не по ответу человека. Даже ответы Baseline можно разделить по `hint_wrong`: эти люди подсказку не видели, но выполняли те же карточки, поэтому Baseline задает контрольный уровень сложности страниц.

### Какие четыре числа нужны

| Срез карточек | Baseline | AI hint | Что сравниваем |
|---|---:|---:|---|
| Подсказка верна | Доля правильных ответов без подсказки | Доля правильных ответов с верной подсказкой | $\Delta Q_{\mathrm{correct}}$ — помощь верной подсказки |
| Подсказка неверна | Доля правильных ответов без подсказки | Доля правильных ответов с неверной подсказкой | $\Delta Q_{\mathrm{wrong}}$ — вред неверной подсказки |

$$\Delta Q_{\mathrm{correct}}=100\,(Q_{\mathrm{AI},\,correct}-Q_{\mathrm{Baseline},\,correct})\quad\text{п.п.}$$

$$\Delta Q_{\mathrm{wrong}}=100\,(Q_{\mathrm{AI},\,wrong}-Q_{\mathrm{Baseline},\,wrong})\quad\text{п.п.}$$

Положительный эффект означает рост правильности после показа подсказки, отрицательный — снижение.

### Практика №3

1. Создайте срез с верной подсказкой и срез с неверной.
2. ...

In [ ]:
# 1. Делим ответы по качеству подсказки, а не по ответу человека.
correct_hint_answers = data.loc[
    ...
]

wrong_hint_answers = data.loc[
    ...
]

# 2. В каждом срезе считаем долю правильных ответов по двум группам.
accuracy_good = (
    correct_hint_answers
    .groupby("condition")["correct"]
    .mean()
    .reindex(ARMS)
)

accuracy_bad = (
    wrong_hint_answers
    .groupby("condition")["correct"]
    .mean()
    .reindex(ARMS)
)

quality_table = pd.DataFrame(
    [accuracy_good, accuracy_bad],
    index=["Верная подсказка", "Неверная подсказка"],
)[ARMS]

# 3. Внутри каждого среза вычитаем Baseline из AI hint.
help_effect_pp = 100 * (
    ...
)

harm_effect_pp = 100 * (
    ...
)

quality_delta_pp = pd.Series({
    "Верная подсказка": help_effect_pp,
    "Неверная подсказка": harm_effect_pp,
})

display(
    quality_delta_pp.to_frame("AI hint − Baseline, п.п.")
)


In [ ]:
quality_labels = [
    f"Верная подсказка\n{hint_is_correct.mean():.0%} карточек",
    f"Неверная подсказка\n{(~hint_is_correct).mean():.0%} карточек",
]
quality_display = quality_table.copy()
quality_display.index = quality_labels
display(quality_display.rename(columns=dict(zip(ARMS,LABELS))).style.format("{:.1%}"))

fig, axes = plt.subplots(1,2,figsize=(12,4),layout="constrained")
(quality_display.rename(columns=dict(zip(ARMS,LABELS))) * 100).plot.bar(ax=axes[0],color=COLORS,rot=0)
axes[0].set(ylabel="Правильных ответов", ylim=(0,110), title="Что происходит с качеством людей?")
axes[0].yaxis.set_major_formatter(PercentFormatter(100)); axes[0].grid(axis="y",alpha=.25)
for bars in axes[0].containers:
    axes[0].bar_label(bars,labels=[f"{v:.1f}%" for v in bars.datavalues],padding=3)
quality_delta_pp.rename(index=dict(zip(quality_table.index,quality_labels))).plot.barh(
    ax=axes[1],color=["#3A9C89","#D76D67"])
axes[1].axvline(0,color="black",lw=1)
axes[1].set(xlabel="AI hint − Baseline, п.п.",title="Изменение правильности")
axes[1].grid(axis="x",alpha=.25)
plt.show()

## Общий результат

$$\Delta Q=w_{\mathrm{correct}}\Delta Q_{\mathrm{correct}}
+w_{\mathrm{wrong}}\Delta Q_{\mathrm{wrong}}.$$


In [ ]:
weights = pd.Series(
    [hint_is_correct.mean(), 1 - hint_is_correct.mean()],
    index=quality_table.index,
)
contributions_pp = weights * quality_delta_pp
decomposition = pd.DataFrame({
    "Доля карточек": weights,
    "Эффект, п.п.": quality_delta_pp,
    "Вклад, п.п.": contributions_pp,
})
display(decomposition.style.format({
    "Доля карточек": "{:.0%}",
    "Эффект, п.п.": "{:+.1f}",
    "Вклад, п.п.": "{:+.1f}",
}))

overall_accuracy = data.groupby("condition")["correct"].mean().reindex(ARMS)
overall_delta_pp = 100 * (overall_accuracy["assisted"] - overall_accuracy["baseline"])
balance_pp = pd.Series({
    "Вклад верных подсказок": contributions_pp["Верная подсказка"],
    "Вклад неверных подсказок": contributions_pp["Неверная подсказка"],
    "Общий результат": contributions_pp.sum(),
})
fig, ax = plt.subplots(figsize=(9, 3.4), layout="constrained")
bars = ax.barh(balance_pp.index, balance_pp, color=["#3A9C89", "#D76D67", "#6479AF"])
ax.bar_label(bars, labels=[f"{value:+.1f} п.п." for value in balance_pp],
             label_type="center", color="white", fontweight="bold")
ax.axvline(0, color="black", lw=1); ax.grid(axis="x", alpha=.25)
ax.set(xlabel="Изменение правильности, п.п.", title="Баланс помощи и вреда")
plt.show()
display(Markdown(
    f"**Все ответы:** {overall_accuracy['baseline']:.1%} → {overall_accuracy['assisted']:.1%} "
    f"({overall_delta_pp:+.1f} п.п.)."
))

## Согласованность полных ответов

**Вопрос:** делает ли общая AI-подсказка ответы трех исполнителей более согласованными?

После удаления подозрительного исполнителя на 10 карточках осталось неполное перекрытие: для анализа согласованности и majority vote исключаем эти карточки и используем 90 карточек, где в обеих группах сохранилось ровно по три ответа.

`1/3` — все ответы различаются, `2/3` — совпали двое, `3/3` — совпали все трое.
Числовые цены сравниваются до копейки, отказы — по статусу.

In [ ]:
# Для согласованности нужны ровно три ответа в обеих группах.
responses_per_card = (
    data.groupby(["sample_id", "condition"])
    .size()
    .unstack()
    .reindex(columns=ARMS)
)
complete_agreement_cards = responses_per_card.index[
    responses_per_card.eq(3).all(axis=1)
]
agreement_data = data.loc[
    data["sample_id"].isin(complete_agreement_cards)
].copy()

excluded_agreement_cards = len(cards) - len(complete_agreement_cards)
print(
    f"Исключили {excluded_agreement_cards} карточек с неполным перекрытием; "
    f"для согласованности осталось {len(complete_agreement_cards)} карточек"
)

agreement_rows = []
for (condition, sample_id), group in agreement_data.groupby(["condition", "sample_id"]):
    answer_keys = []
    for row in group.itertuples():
        if row.numeric:
            answer_key = f"price:{int(round(row.price * 100))}"
        else:
            answer_key = f"status:{row.status}"
        answer_keys.append(answer_key)

    largest_group = pd.Series(answer_keys).value_counts().iloc[0]
    agreement_rows.append({
        "condition": condition,
        "sample_id": sample_id,
        "agreement": f"{largest_group}/3",
    })

agreement_by_card = pd.DataFrame(agreement_rows)
agreement_levels = ["1/3", "2/3", "3/3"]
agreement_shares = pd.crosstab(
    agreement_by_card["condition"],
    agreement_by_card["agreement"],
    normalize="index",
).reindex(index=ARMS, columns=agreement_levels, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4))
plot_data = 100 * agreement_shares.rename(index=dict(zip(ARMS, LABELS))).T
plot_data.plot.bar(ax=ax, color=COLORS, rot=0)
for bars in ax.containers:
    ax.bar_label(
        bars,
        labels=[f"{value:.0f}%" for value in bars.datavalues],
        padding=3,
    )
ax.set(
    xlabel="Максимальное число совпавших ответов",
    ylabel="Доля карточек",
    ylim=(0, 100),
    title="Согласованность трех исполнителей",
)
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.grid(axis="y", alpha=.25)
plt.show()

### От согласованности к энтропии

Для одной карточки считаем доли каждого уникального ответа среди трех исполнителей:

$$H(p)=-\sum_i p_i\log p_i.$$

Три уровня согласованности дают три значения:

$$H_{3/3}=-(1\log 1)=0,$$

$$H_{2/3}=-\left(\frac23\log\frac23+\frac13\log\frac13\right)\approx0{,}64,$$

$$H_{1/3}=-3\left(\frac13\log\frac13\right)=\log3\approx1{,}10.$$

То есть путь **«все совпали → совпали двое → все различаются»** соответствует росту неопределённости **$0 → 0{,}64 → 1{,}10$**.

Средняя энтропия группы — взвешенная сумма этих уровней с долями карточек из предыдущего графика:

$$\bar H_{\mathrm{Baseline}}\approx0{,}07\cdot1{,}10+0{,}39\cdot0{,}64+0{,}54\cdot0=0{,}32,$$

$$\bar H_{\mathrm{AI\ hint}}\approx0{,}02\cdot1{,}10+0{,}31\cdot0{,}64+0{,}67\cdot0=0{,}22.$$

Здесь `39%` и `31%` — доли карточек с согласованностью `2/3`, а `0,64` — энтропия одной такой карточки.

In [ ]:
# Энтропия измеряет разброс трёх ответов, но не их правильность.
entropy_by_agreement = pd.Series({
    "1/3": np.log(3),
    "2/3": -(2 / 3 * np.log(2 / 3) + 1 / 3 * np.log(1 / 3)),
    "3/3": 0.0,
})
mean_entropy = agreement_shares @ entropy_by_agreement

fig, ax = plt.subplots(figsize=(8, 2.8), layout="constrained")
bars = ax.barh(LABELS, mean_entropy.reindex(ARMS), color=COLORS)
ax.bar_label(bars, labels=[f"{value:.2f}" for value in mean_entropy.reindex(ARMS)], padding=4)
ax.set(xlabel="Средняя энтропия ответов на карточке", xlim=(0, np.log(3)),
       title="Насколько распределены ответы исполнителей?")
ax.set_xticks(
    [0, entropy_by_agreement["2/3"], entropy_by_agreement["1/3"]],
    ["все совпали", "совпали двое", "все различаются"],
)
ax.grid(axis="x", alpha=.25); ax.invert_yaxis(); plt.show()

display(Markdown(
    f"**Средняя энтропия снизилась: {mean_entropy['baseline']:.2f} → {mean_entropy['assisted']:.2f}.** "
    "Чем выше энтропия, тем больше разных вариантов и тем меньше мы знаем о доминирующем ответе. "
    "Здесь ответы стали более сконцентрированными: общая подсказка чаще направляет людей к одному варианту. "
    "Но энтропия не проверяет правильность — три одинаковых неверных ответа тоже дают нулевую энтропию."
))

## Majority vote

Используем те же 90 карточек с полным перекрытием: по три ответа в обеих группах.

Согласованность выросла, но общий ответ мог стать как верным, так и неверным.

**Следующий вопрос:** улучшился ли результат большинства после того, как ответы стали более согласованными?

In [ ]:
def majority_price(group):
    numeric_prices = group.loc[group["numeric"], "price"]
    prices_in_kopecks = (numeric_prices * 100).round().astype(int)
    vote_counts = prices_in_kopecks.value_counts()

    if vote_counts.empty or vote_counts.iloc[0] < 2:
        return np.nan

    return vote_counts.index[0] / 100


majority_rows = []
for (condition, sample_id), group in agreement_data.groupby(["condition", "sample_id"]):
    majority_rows.append({
        "condition": condition,
        "sample_id": sample_id,
        "majority_price": majority_price(group),
    })

majority_answers = pd.DataFrame(majority_rows).merge(
    cards[["sample_id", "reference_price"]],
    on="sample_id",
    validate="many_to_one",
)
majority_answers["correct"] = (
    majority_answers["majority_price"]
    - majority_answers["reference_price"]
).abs() < .005
majority_accuracy = (
    majority_answers.groupby("condition")["correct"]
    .mean()
    .reindex(ARMS)
)


In [ ]:
majority_answers["outcome"] = np.select(
    [majority_answers["correct"],majority_answers["majority_price"].isna()],
    ["Верное большинство","Нет большинства"],default="Ошибочное большинство")
majority_shares = pd.crosstab(majority_answers.condition,majority_answers.outcome,normalize="index")
majority_shares = majority_shares.reindex(index=ARMS,columns=["Верное большинство","Ошибочное большинство","Нет большинства"],fill_value=0)
np.testing.assert_allclose(majority_shares["Верное большинство"],majority_accuracy)
ax = majority_shares.rename(index=dict(zip(ARMS,LABELS))).plot.barh(
    stacked=True,color=["#3A9C89","#D76D67","#CBD1DD"],figsize=(9,3))
ax.set(xlim=(0,1),xlabel="Доля карточек",ylabel="",title="Что получилось после голосования?")
ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.grid(axis="x",alpha=.2)
ax.legend(loc="upper center",bbox_to_anchor=(.5,-.25),ncols=3,frameon=False)
plt.show()
display(Markdown("**Ответ:** большинство чаще верно с подсказкой, но ошибочное большинство тоже встречается чаще."))


## Чувствительность результата к качеству подсказок

Рассмотрим закон полного математического ожидания. Если событие $C$ означает «подсказка верна», событие $W$ — «подсказка неверна», а $p=P(C)$, то:

$$E[\Delta Q]=P(C)E[\Delta Q\mid C]+P(W)E[\Delta Q\mid W].$$

Обозначим:

- $Q_{\mathrm{AI},C}$ — качество AI hint на карточках с верной подсказкой;
- $Q_{\mathrm{Base},C}$ — качество Baseline на тех же карточках;
- $Q_{\mathrm{AI},W}$ и $Q_{\mathrm{Base},W}$ — аналогично для карточек с неверной подсказкой.

На предыдущем шаге мы оценили два условных различия между группами:

$$\Delta Q_C=100\,(Q_{\mathrm{AI},C}-Q_{\mathrm{Base},C})=+17{,}7\text{ п.п.},$$

$$\Delta Q_W=100\,(Q_{\mathrm{AI},W}-Q_{\mathrm{Base},W})=-12{,}8\text{ п.п.}$$

При доле верных подсказок $p$ доля неверных равна $1-p$. Подставляем эти два условных эффекта в закон полного ожидания:

$$E(p)=p\Delta Q_C+(1-p)\Delta Q_W.$$

Точка безубыточности получается из $E(p)=0$:

$$p^*=\frac{-\Delta Q_W}{\Delta Q_C-\Delta Q_W}=\frac{12{,}8}{17{,}7+12{,}8}\approx42{,}1\%.$$

Для текущего состава, где верны $67\%$ подсказок:

$$E(0{,}67)=0{,}67\cdot17{,}7+0{,}33\cdot(-12{,}8)\approx+7{,}6\text{ п.п.}$$


> **Модельное допущение:** это сценарный расчет для текущего эксперимента, а не универсальный прогноз. Мы фиксируем измеренные здесь эффекты $+17{,}7$ и $-12{,}8$ п.п. и меняем только долю верных подсказок. Если улучшение модели изменит набор карточек, на которых она ошибается, поведение людей и сами условные эффекты тоже могут измениться. Тогда их нужно оценить заново.

### Практика №4

1. ...

In [ ]:
# TODO: найдите эффекты на предыдущем графике.
help_effect_pp = ...
harm_effect_pp = ...

def expected_quality_effect(hint_accuracy):
    # TODO: взвесьте помощь и вред.
    return ...

break_even_accuracy = (
    -harm_effect_pp
    / (help_effect_pp - harm_effect_pp)
)

print(
    "Точка безубыточности:",
    f"{break_even_accuracy:.1%} верных подсказок",
)


In [ ]:
hint_accuracy_grid = np.linspace(0, 1, 101)
expected_effect_grid = [expected_quality_effect(value) for value in hint_accuracy_grid]
current_hint_accuracy = hint_is_correct.mean()
current_effect_pp = expected_quality_effect(current_hint_accuracy)

fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
ax.axvspan(0, break_even_accuracy, color="#D76D67", alpha=.12)
ax.axvspan(break_even_accuracy, 1, color="#3A9C89", alpha=.12)
ax.plot(hint_accuracy_grid, expected_effect_grid, color="#3F5D9B", linewidth=3)
ax.scatter([0, 1], [harm_effect_pp, help_effect_pp], color="#3F5D9B", s=70, zorder=3)
ax.axhline(0, color="black", linewidth=1)
ax.axvline(break_even_accuracy, color="black", linestyle="--", linewidth=1)
ax.scatter(current_hint_accuracy, current_effect_pp, color="#E1874F", s=90, zorder=4)

ax.annotate(f"0% верных: {harm_effect_pp:+.1f} п.п.", (0, harm_effect_pp),
            xytext=(10, 12), textcoords="offset points")
ax.annotate(f"100% верных: {help_effect_pp:+.1f} п.п.", (1, help_effect_pp),
            xytext=(-10, -18), textcoords="offset points", ha="right")
ax.annotate(f"{break_even_accuracy:.1%} — польза = вред", (break_even_accuracy, 0),
            xytext=(10, 12), textcoords="offset points")
ax.annotate(f"Текущий состав: {current_hint_accuracy:.0%}\nэффект {current_effect_pp:+.1f} п.п.",
            (current_hint_accuracy, current_effect_pp), xytext=(12, 12), textcoords="offset points")
ax.set(title="Сценарий при неизменных эффектах верной и неверной подсказки",
       xlabel="Доля верных подсказок", ylabel="Изменение правильности, п.п.",
       xlim=(0, 1), ylim=(harm_effect_pp - 3, help_effect_pp + 3))
ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.grid(alpha=.25); plt.show()

### Как строится график

1. По горизонтали откладываем долю верных подсказок $p$ от `0%` до `100%`.
2. По вертикали — ожидаемое изменение правильности людей $E(p)$ в процентных пунктах.
3. При $p=0$ формула дает $E(0)=\Delta Q_W=-12{,}8$ п.п.: в сценарии остаётся только измеренный вред неверной подсказки.
4. При $p=1$ получаем $E(1)=\Delta Q_C=+17{,}7$ п.п.: остаётся только измеренная помощь верной подсказки. 

> Это края сценарной модели, а не два отдельно проведенных эксперимента со 100% неверной и 
100% верной моделью.

5. Раскрываем скобки: $E(p)=\Delta Q_W+p(\Delta Q_C-\Delta Q_W)$. Поэтому при зафиксированных условных эффектах зависимость линейна и две крайние точки соединяются прямой.
6. Пересечение прямой с нулем — точка безубыточности: помощь и вред компенсируют друг друга.
7. Оранжевая точка показывает фактический состав текущей выборки: $67%$ верных подсказок и сценарный эффект $+7{,}6$ п.п.

**Вывод:** в рамках этой сценарной модели точка безубыточности равна $42{,}1%$. Текущие $67%$ находятся выше порога, поэтому ожидаемая помощь перевешивает вред. Но порог нельзя автоматически переносить на другую модель или другой набор страниц: расчет верен только пока условные эффекты считаются неизменными.

## Границы результатов

Три непосредственно наблюдаемых уровня качества:

- только модель: $67{,}0\%$ верных подсказок;
- человек без подсказки: $71{,}7\%$ правильных ответов;
- человек с подсказкой: $79{,}3\%$ правильных ответов.

$$79{,}3\%>71{,}7\%>67{,}0\%.$$

На представленном наборе у связки «человек + AI hint» самая высокая точечная оценка качества.

In [ ]:
from matplotlib.patches import FancyBboxPatch

time_values = median_time_filtered.reindex(ARMS)
fig, ax = plt.subplots(figsize=(12, 6.5), layout="constrained")
ax.set(xlim=(0, 1), ylim=(0, 1)); ax.axis("off")
ax.text(.5, .96, "Что изменилось после добавления AI-подсказки?",
        ha="center", va="center", fontsize=17, fontweight="bold")
ax.text(.38, .89, "Baseline", ha="center", color="#6B7280", fontweight="bold")
ax.text(.61, .89, "AI hint", ha="center", color="#4E648B", fontweight="bold")
ax.text(.84, .89, "Результат", ha="center", fontweight="bold")

rows = [
    dict(y=.76, title="Общее качество",
         baseline=f"{overall_accuracy['baseline']:.1%}", assisted=f"{overall_accuracy['assisted']:.1%}",
         effect=f"{overall_delta_pp:+.1f} п.п."),
    dict(y=.57, title="Верная подсказка",
         baseline=f"{quality_table.loc['Верная подсказка', 'baseline']:.1%}",
         assisted=f"{quality_table.loc['Верная подсказка', 'assisted']:.1%}",
         effect=f"{quality_delta_pp['Верная подсказка']:+.1f} п.п."),
    dict(y=.38, title="Неверная подсказка",
         baseline=f"{quality_table.loc['Неверная подсказка', 'baseline']:.1%}",
         assisted=f"{quality_table.loc['Неверная подсказка', 'assisted']:.1%}",
         effect=f"{quality_delta_pp['Неверная подсказка']:+.1f} п.п."),
    dict(y=.19, title="Время",
         baseline=f"{time_values['baseline']:.1f} с", assisted=f"{time_values['assisted']:.1f} с",
         effect=f"{time_after_filter['effect']:+.1%}"),
]

for row in rows:
    ax.add_patch(FancyBboxPatch((.04, row['y'] - .075), .92, .15,
                 boxstyle="round,pad=0.012,rounding_size=0.025",
                 facecolor="#F3F5F8", edgecolor="#D7DBE3", linewidth=1))
    ax.text(.09, row['y'], row['title'], fontsize=14, fontweight="bold", va="center")
    ax.text(.38, row['y'], row['baseline'], fontsize=18, fontweight="bold", ha="center", va="center")
    ax.annotate("", xy=(.55, row['y']), xytext=(.45, row['y']),
                arrowprops=dict(arrowstyle="-|>", color="#8D96A8", lw=2.5))
    ax.text(.61, row['y'], row['assisted'], fontsize=18, fontweight="bold", ha="center", va="center")
    ax.text(.84, row['y'], row['effect'], fontsize=17, fontweight="bold",
            color="#4E648B", ha="center", va="center")
plt.show()

## Итог

- Убедительного изменения времени не обнаружено: после фильтра оценка равна $+7{,}6\%$, но 95% интервал включает ноль.
- Точечная оценка правильности изменилась с $71{,}7\%$ до $79{,}3\%$, но 95% интервал разницы $[-1{,}9; +17{,}8]$ п.п. включает ноль.
- У связки «человек + AI hint» самая высокая точечная оценка на подготовленном наборе.
- Верные подсказки помогали, но неверные снижали качество и направляли людей к конкретной ошибке модели.
- В сценарном расчете при неизменных условных эффектах помощь начинает перевешивать вред после $42{,}1\%$ верных подсказок.

### Материалы

- Лекция 5: качество и время, α Криппендорфа, uptake и влияние подсказок, независимый контроль.
- [Roitero et al., 2025](https://arxiv.org/abs/2501.18265).
- [Варианты расстояния для α](https://arxiv.org/abs/2103.12170).